# 01 · Data inspection
**Question.** What exactly is in the raw file, and is it a clean, regular hourly time series?
**Scope.** The raw file only: no split, no modelling and no statistic that feeds a design decision.
**Code.** `src/data.py` (loading, datetime, timeline validation, inspection). Live cells below; the stored tables come from `python run_all.py`.

The file is `PRSA_Data_Aotizhongxin_20130301-20170228.csv` from the notebook the project started from. The provenance note printed below explains why it is
attributed to the UCI *Beijing Multi-Site Air-Quality Data* set (id 501) and not to the `fetch_ucirepo(id=381)` snippet of the assignment brief.

In [1]:
import io, json, re, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from PIL import Image as PILImage
from IPython.display import Image, Markdown, display
from src import config as C

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)

def table(stem, **kw):
    """results/tables/<stem>.csv (written by run_all.py)."""
    return pd.read_csv(C.TABLES / f"{stem}.csv", **kw)

def caption(stem):
    txt = (C.FIGURES / "figure_captions.md").read_text()
    m = re.search(rf"\*\*{re.escape(stem)}\*\* - (.*?)(?:\n\n|\Z)", txt, re.S)
    return m.group(1).strip() if m else ""

def fig(stem, width=1500):
    """Show results/figures/<stem>.png (down-scaled to keep the notebook small) with the caption computed from the data."""
    im = PILImage.open(C.FIGURES / f"{stem}.png")
    if im.width > width:
        im = im.resize((width, round(im.height * width / im.width)), PILImage.LANCZOS)
    buf = io.BytesIO(); im.save(buf, format="PNG", optimize=True)
    display(Image(data=buf.getvalue())); display(Markdown(f"*{caption(stem)}*"))

def log_grep(pattern, maxlen=420):
    """Print the lines of results/logs/run_all.log that match a regular expression."""
    for line in (C.LOGS / "run_all.log").read_text().splitlines():
        if re.search(pattern, line):
            print(line[:maxlen])

## 1. Source, size and schema

In [2]:
from src import data as D
raw = D.load_raw()                                   # asserts the expected number of rows
prov = D.provenance()
print(f"{prov['name']}  (DOI {prov['doi']})")
print(f"station: {prov['station']} | file: {prov['file']} | sha256 {prov['sha256'][:16]}...")
print("note   :", prov["note"])
display(raw.head())
rep = D.inspect_raw(raw)
print(f"\nrows: {rep['rows']:,} | columns: {rep['columns']}")
display(rep["dtypes"].round({"missing_pct": 2}))
print("duplicated rows:", rep["duplicate_rows"], "| duplicated rows ignoring the 'No' counter:", rep["duplicate_rows_ignoring_No"])
print("wind-direction levels (including NaN):", len(rep["wd_levels"]))

Beijing Multi-Site Air-Quality Data (UCI Machine Learning Repository, id 501)  (DOI 10.24432/C5RK5G)
station: Aotizhongxin | file: PRSA_Data_Aotizhongxin_20130301-20170228.csv | sha256 a94fbcfc71708b6f...
note   : The snippet fetch_ucirepo(id=381) returns a DIFFERENT dataset ('Beijing PM2.5', 43,824 rows, US-embassy station, columns cbwd/Iws/Is/Ir, no PM10/SO2/NO2/CO/O3/RAIN/WSPM). The Aotizhongxin file belongs to id 501, which ucimlrepo cannot import.


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin



rows: 35,064 | columns: 18


,column,dtype,non_null,missing,missing_pct
0,No,int64,35064,0,0.00
1,year,int64,35064,0,0.00
2,month,int64,35064,0,0.00
3,day,int64,35064,0,0.00
4,hour,int64,35064,0,0.00
5,PM2.5,float64,34139,925,2.64
6,PM10,float64,34346,718,2.05
7,SO2,float64,34129,935,2.67
8,NO2,float64,34041,1023,2.92
9,CO,float64,33288,1776,5.07


duplicated rows: 0 | duplicated rows ignoring the 'No' counter: 0
wind-direction levels (including NaN): 17


## 2. Timeline checks (asserted, not assumed)
`year/month/day/hour` → `datetime`, sorted chronologically; `validate_timeline` asserts that the series is monotonic, has no duplicated timestamps, a single sampling
interval of one hour and no missing timestamp. `No` (row counter) and `station` (constant) are identifiers and are dropped.

In [3]:
df, was_sorted = D.build_datetime(raw)
tl = D.validate_timeline(df)
print("already sorted in the file:", was_sorted)
display(pd.Series(tl, name="value").to_frame())
df, ids = D.drop_identifier_columns(df)
print("identifier columns removed:", ids)

already sorted in the file: True


,value
monotonic_increasing,True
duplicated_timestamps,0
sampling_intervals,{'0 days 01:00:00': 35063}
start,2013-03-01 00:00:00
end,2017-02-28 23:00:00
expected_hourly_stamps,35064
observed_stamps,35064
missing_timestamps,0
span_days,1460.96


identifier columns removed: {'dropped': ['No', 'station'], 'constant_values': {'station': ['Aotizhongxin']}}


## 3. Target statistics (descriptive, whole file)
A structural summary only. Every distribution used for a modelling decision is computed on the **training rows** in notebook 03.

In [4]:
display(pd.Series(rep["target_stats"], name="PM2.5 (µg/m³), whole file").to_frame().T.round(2))

,count,missing,mean,std,min,q25,median,q75,max,skew
"PM2.5 (µg/m³), whole file",34139.0,925.0,82.77,82.14,3.0,22.0,58.0,114.0,898.0,1.97


## 4. Stored tables (`results/tables`)

In [5]:
display(table("table_1_dataset_statistics"))
display(table("table_1b_target_statistics").round(1))
display(table("table_S01_schema_and_missing_values").round(2))

,item,value
0,Source,"UCI 'Beijing Multi-Site Air-Quality Data' (id 501), DOI 10.24432/C5RK5G; station Aotiz..."
1,Rows x columns (file),"35,064 x 18 (hourly, 2013-03-01 to 2017-02-28; 1460.96 days)"
2,Sampling,"hourly, regular (0 missing / 0 duplicated timestamps)"
3,Dropped identifier columns,"No (row counter), station (constant)"
4,Missing values (whole file),"PM2.5 925, PM10 718, SO2 935, NO2 1023, CO 1776, O3 1719, TEMP 20, PRES 20, DEWP 20, R..."
5,Task,predict PM2.5 at t+1 (µg/m³) from information available at hour t
6,Train / test split (rows),"28,050 / 7,013 hours (2013-03-01 to 2016-05-12 | 2016-05-12 to 2017-02-28)"
7,Usable rows (train / test),"26,000 / 6,752 (missing target, missing current PM2.5 or unusable history excluded)"
8,Candidate features / Set A / Set C (design columns),37 / 26 / 54


,split,n,mean,std,min,q25,median,q75,max,skew
0,train (usable rows),26000,83.4,81.2,3.0,23.0,60.0,116.0,898.0,1.9
1,test (usable rows),6752,80.7,84.6,3.0,20.0,54.0,109.0,713.0,2.1


,column,dtype,non_null,missing,missing_pct
0,No,int64,35064,0,0.00
1,year,int64,35064,0,0.00
2,month,int64,35064,0,0.00
3,day,int64,35064,0,0.00
4,hour,int64,35064,0,0.00
5,PM2.5,float64,34139,925,2.64
6,PM10,float64,34346,718,2.05
7,SO2,float64,34129,935,2.67
8,NO2,float64,34041,1023,2.92
9,CO,float64,33288,1776,5.07
